# It Takes Time: Temporal Collaborative Filtering Benchmark

Goal: rank a battery of sequential / temporal collaborative-filtering models on
**MovieLens-1M** under a single, reproducible methodology, and produce a results
table that's easy to extend with custom variants.

**Models compared (v1)**

| Family | Models |
|---|---|
| Non-sequential baselines | `Pop`, `BPR`, `ItemKNN` |
| Sequential SOTA | `FPMC`, `GRU4Rec`, `NARM`, `SASRec`, `BERT4Rec` |

**Methodology**

- Dataset: MovieLens-1M, filtered to users / items with ≥ 5 interactions.
- Split: leave-one-out per user, sorted chronologically (RecBole `LS=valid_and_test`, `order=TO`).
  Last interaction → test, second-to-last → validation, the rest → train.
- HPO: Optuna with TPE sampler + MedianPruner. `N_TRIALS=5` by default —
  scale up by setting the `N_TRIALS` env var.
- Eval: full-vocabulary scoring (`mode='full'`) → HitRate, NDCG, MRR, Recall, Precision @ {10, 20, 50, 100}.
- Resumability: each model's eval result is persisted as JSON under
  `results/eval/<dataset>__<model>.json`. Re-running the benchmark cell skips any
  model whose JSON already exists. Optuna studies persist to SQLite under
  `results/hpo/`, so HPO trials also resume.


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import optuna

import config
from data import prepare_recbole_dataset, dataset_stats
from models import MODEL_REGISTRY, list_models
from hpo import run_optuna
from runner import train_and_eval, has_result, load_result
from evaluation import aggregate_results, top_k_recommend

print('device:', config.DEVICE)
print('models:', list_models())
print('N_TRIALS:', config.N_TRIALS, '| HPO_EPOCHS:', config.HPO_EPOCHS, '| FINAL_EPOCHS:', config.FINAL_EPOCHS)

## 2. Dataset preparation

Downloads MovieLens-1M (~6 MB) on first run, converts to RecBole atomic-file
format, and caches under `recbole_data/ml-1m/ml-1m.inter`. Subsequent runs are no-ops.

In [ ]:
DATASET = 'ml-1m'
prepare_recbole_dataset(DATASET)
stats = dataset_stats(DATASET)
pd.Series(stats).to_frame('value')

## 3. Run benchmark (resumable)

Loops the model registry. For each model:

1. If `results/eval/<dataset>__<model>.json` exists → load it, skip training.
2. Otherwise: run Optuna HPO, then a final fit on the best params, persist JSON.

Interrupt at any point; re-running this cell continues from the next un-evaluated model.

In [ ]:
MODELS_TO_RUN = list_models()  # or e.g. ['Pop', 'BPR', 'SASRec']

results = {}
for name in MODELS_TO_RUN:
    print(f'\n=== {name} ===')
    if has_result(DATASET, name):
        results[name] = load_result(DATASET, name)
        print(f'  cached → NDCG@10={results[name]["test_result"].get("ndcg@10"):.4f}')
        continue
    hpo_out = run_optuna(DATASET, name)
    print(f'  HPO best_value={hpo_out["best_value"]} params={hpo_out["best_params"]}')
    results[name] = train_and_eval(DATASET, name, best_params=hpo_out['best_params'])
    print(f'  final NDCG@10={results[name]["test_result"].get("ndcg@10"):.4f}')

print('\nAll done.')

## 4. Results table

In [ ]:
df = aggregate_results(DATASET)
key_cols = ['model', 'hit@10', 'ndcg@10', 'mrr@10', 'recall@10', 'hit@100', 'ndcg@100', 'train_seconds']
key_cols = [c for c in key_cols if c in df.columns]
df[key_cols].round(4)

In [ ]:
import matplotlib.pyplot as plt
ax = df.set_index('model')[[c for c in ['ndcg@10', 'ndcg@20', 'ndcg@50', 'ndcg@100'] if c in df.columns]].plot.bar(figsize=(10, 4))
ax.set_ylabel('NDCG'); ax.set_title('NDCG@K by model'); plt.xticks(rotation=30, ha='right'); plt.tight_layout()

## 5. Top-100 recommendations demo

Confirms the "next-1 → top-K" path: we score the entire item vocab at the next
position, mask items already in the user's history, and take the top 100. The
model below is reloaded from its checkpoint.

In [ ]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed

best_model_name = df.iloc[0]['model']
print('Best model:', best_model_name)
rec = load_result(DATASET, best_model_name)
ckpt = rec['checkpoint']

# Rebuild config + dataset and load checkpoint
from runner import _build_config
cfg, model_cls = _build_config(DATASET, best_model_name, rec.get('best_params') or {}, epochs=1, saved=True)
init_seed(cfg['seed'], cfg['reproducibility'])
dataset = create_dataset(cfg)
train_data, valid_data, test_data = data_preparation(cfg, dataset)
model_class = model_cls or get_model(cfg['model'])
model = model_class(cfg, train_data._dataset).to(cfg['device'])
model.load_state_dict(torch.load(ckpt, map_location=cfg['device'])['state_dict'])
model.eval()

sample_users = list(dataset.id2token(dataset.uid_field, np.arange(1, 4)))
topk = top_k_recommend(model, dataset, test_data, sample_users, k=100)
topk.head(15)

## 6. Analysis

In [ ]:
# Quick sequential-vs-non-sequential gap
is_seq = {n: (MODEL_REGISTRY[n]['type'] == 'sequential') for n in df['model']}
df_anal = df.copy()
df_anal['family'] = df_anal['model'].map(lambda n: 'sequential' if is_seq[n] else 'general')
df_anal.groupby('family')[[c for c in ['ndcg@10', 'hit@10', 'mrr@10'] if c in df_anal.columns]].mean().round(4)

## 7. Adding a custom variant

For your next paper, drop a class into [`src/models/variants/`](../src/models/variants/)
and register it. Example:

```python
# src/models/variants/sasrec_plus.py
from recbole.model.sequential_recommender.sasrec import SASRec
import torch.nn as nn

class SASRecPlus(SASRec):
    def __init__(self, config, dataset):
        super().__init__(config, dataset)
        self.extra_proj = nn.Linear(self.hidden_size, self.hidden_size)
    def forward(self, item_seq, item_seq_len):
        out = super().forward(item_seq, item_seq_len)
        return out + self.extra_proj(out)  # toy modification
```

Then in [`src/models/__init__.py`](../src/models/__init__.py):

```python
from models.variants.sasrec_plus import SASRecPlus
MODEL_REGISTRY['SASRecPlus'] = {
    'class': SASRecPlus,
    'type': 'sequential',
    'search_space': sasrec_space,
    'static': _seq_static(),
}
```

Re-run section 3 — only `SASRecPlus` will train; everything else loads from disk.